# Análise Exploratória e Demonstração do Pipeline de Sentimentos

Este notebook serve como recurso secundário para exploração interativa dos dados e dos modelos do projeto `sentiment_transformer`.

**Importante**: A lógica científica principal do projeto reside no pacote Python `sentiment_transformer` e pode ser executada via CLI ou scripts em `scripts/`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Adicionar diretório src ao path caso execute fora do pacote instalado
project_root = Path("..").resolve()
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

from sentiment_transformer.config import load_config
from sentiment_transformer.data.audit import audit_dataset
from sentiment_transformer.modeling.device import get_device_info
from sentiment_transformer.constants import (
    COL_COMBINED_TEXT,
    COL_LABEL,
    COL_OVERALL_RATING,
    RATING_3_EXCLUSION_JUSTIFICATION,
)

config = load_config("../configs/default.yaml")
print(f"Configuração carregada: {config.project.name}")
print(f"Modelo Transformer: {config.transformer.model_checkpoint}")

## 1. Auditoria do Ambiente e Hardware

In [ ]:
device_info = get_device_info()
print(f"Sistema Operacional: {device_info['os_name']} {device_info['os_release']}")
print(f"PyTorch: {device_info['pytorch_version']}")
print(f"CUDA disponível: {device_info['cuda_available']}")
print(f"Memória RAM Total: {device_info['ram_total_gb']} GB")

## 2. Inspeção dos Dados Processados

Verifica se as partições já foram geradas pela fase de dados.

In [ ]:
proc_dir = project_root / "data" / "processed"
train_path = proc_dir / "train.csv"

if train_path.is_file():
    train_df = pd.read_csv(train_path, encoding="utf-8")
    print(f"Partição de treino carregada com {len(train_df)} registros.")
    print("\nDistribuição de Rótulos no Treino:")
    print(train_df[COL_LABEL].value_counts().rename(index={0: "Negativa (0)", 1: "Positiva (1)"}))
    print("\nExemplos de texto combinado (título + [SEP] + corpo):")
    for i, txt in enumerate(train_df[COL_COMBINED_TEXT].head(3), 1):
        print(f"[{i}] {txt}\n")
else:
    print("Partições de dados ainda não geradas.")
    print("Execute: python scripts/prepare_data.py --config configs/smoke.yaml")

## 3. Justificativa Metodológica da Exclusão da Nota 3

A nota 3 é excluída no experimento binário para evitar ruído de rotulagem fraca.

In [ ]:
print(RATING_3_EXCLUSION_JUSTIFICATION)